# Spectra v3: Per-Cluster CN-Aware Autoencoder for LAMOST CN-Star Detection

**Core Hypothesis**: Different stellar clusters have different spectral characteristics 
(Teff/logg/Feh distributions, CN abundance baselines). Training one global AE forces 
a "compromise" representation that may miss cluster-specific CN signatures.

**Approach**: Fine-tune a global CN-aware AE on each cluster separately, producing 
cluster-conditioned encoders that better capture within-cluster CN variations.

**Pipeline**: Global CN-aware pretraining → Per-cluster fine-tuning → 
Cluster-conditioned feature extraction → PU-Bagging → Comparison with global AE & raw spectra

**Experiments**:
- Latent dims: 64d, 128d, 256d
- Band weight: 5.0x (CN3839, CN4142, CH4300)
- Min cluster size: 100 stars (35/45 clusters, 99% of all stars)

**Key Comparisons**:
1. Global CN-aware AE vs Per-cluster CN-aware AE (same dim)
2. Per-cluster AE across dims (64d vs 128d vs 256d)
3. All AEs vs Raw Spectra (upper bound)

**Data**: 33,589 LAMOST spectra (3800-4500A, 700px), 73 known CN-enhanced positives, 45 clusters


In [ ]:
import sys, warnings, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Robust project root detection ──
_NB_DIR = Path().resolve()
_ROOT = _NB_DIR
if _ROOT.name == 'SpectraAE':
    _ROOT = _ROOT.parent
if not (_ROOT / 'ML' / '_cache' / 'X_clean.npy').exists():
    for _candidate in [_NB_DIR.parent, _NB_DIR.parent.parent]:
        if (_candidate / 'ML' / '_cache' / 'X_clean.npy').exists():
            _ROOT = _candidate
            break

if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from SpectraAE.models.autoencoder import ConvAutoencoder, Encoder, Decoder
from SpectraAE.extract_features import extract_features
from SpectraAE.cn_aware_pretrain import (
    create_band_weight_mask, CN_BAND_DEFS,
)
from SpectraAE.cluster_ae import (
    get_cluster_assignments, extract_cluster_features,
)
from ML.pu_bagging import load_pu_data, run_pu_bagging, build_comparison_df
from ML.utils import FEATURE_COLS_CN9, compute_cluster_zscore
from ML.utils import plot_candidate_spectra, plot_teff_logg_distribution

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root: {_ROOT}')
print(f'Device: {DEVICE}')
print(f'Imports OK')


## 1. Cluster Data Analysis

Understanding the cluster distribution is essential for per-cluster AE training.
Each cluster has distinct physical properties (Teff/logg/Feh ranges, CN abundance baselines).


In [ ]:
# Load data
X_clean = np.load(_ROOT / 'ML' / '_cache' / 'X_clean.npy').astype(np.float32)
stars_clean = pd.read_pickle(_ROOT / 'ML' / '_cache' / 'stars_clustered.pkl')

data_pu = load_pu_data()
y_all = data_pu['y_all']
cluster_ids = data_pu['cluster_ids']
df_model = data_pu['df_model']
pos_mask = y_all == 1
n_pos = int(y_all.sum())

unique_clusters, counts = np.unique(cluster_ids, return_counts=True)
n_clusters = len(unique_clusters)

print(f'Spectra: {X_clean.shape}')
print(f'Clusters: {n_clusters}  Stars: {len(cluster_ids):,}  CN+: {n_pos}')
print(f'Size: min={counts.min()}, max={counts.max()}, median={np.median(counts):.0f}, mean={counts.mean():.1f}')

# Cluster size histogram + CN+ overlay
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.barh(range(n_clusters), sorted(counts), height=0.7, color='#3498db', alpha=0.7)
ax.axvline(x=100, color='#e74c3c', ls='--', lw=1.5, label='Min cluster size = 100')
ax.set_xlabel('Stars per Cluster')
ax.set_ylabel('Cluster Index (sorted by size)')
ax.set_title(f'Cluster Size Distribution ({n_clusters} clusters)')
ax.legend()
ax.grid(alpha=0.2, axis='x')

ax = axes[1]
cn_per_cluster = [int(y_all[cluster_ids == c].sum()) for c in unique_clusters]
sizes_sorted = sorted(counts)
for i, (size, n_cn) in enumerate(zip(sizes_sorted, sorted(zip(counts, cn_per_cluster), key=lambda x: x[0]))):
    n_cn = n_cn[1]
    color = '#e74c3c' if n_cn > 0 else '#95a5a6'
    alpha = 0.9 if n_cn > 0 else 0.3
    ax.barh(i, size, height=0.7, color=color, alpha=alpha)
    if n_cn > 0:
        ax.text(size + 10, i, f'CN+={n_cn}', va='center', fontsize=7, color='#e74c3c')
ax.set_xlabel('Stars per Cluster')
ax.set_ylabel('Cluster Index (sorted by size)')
ax.set_title(f'Clusters with CN Positives ({sum(1 for n in cn_per_cluster if n > 0)}/{n_clusters})')
ax.grid(alpha=0.2, axis='x')

fig.suptitle('Cluster Data Overview', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Summary stats
print(f'\nClusters >= 100 stars: {(counts >= 100).sum()}/{n_clusters}')
print(f'Stars in those clusters: {counts[counts >= 100].sum():,} ({counts[counts >= 100].sum()/len(cluster_ids)*100:.1f}%)')
print(f'CN+ in those clusters: {sum(y_all[cluster_ids == c].sum() for c in unique_clusters[counts >= 100])}/{n_pos}')
print(f'\nClusters with CN+: {sum(1 for n in cn_per_cluster if n > 0)}')
for cid, size, n_cn in sorted(zip(unique_clusters, counts, cn_per_cluster), key=lambda x: x[1], reverse=True):
    if n_cn > 0:
        print(f'  Cluster {cid}: {size} stars, {n_cn} CN+')


## 2. Load Models: Global vs Per-Cluster Encoders

For each latent dimension (64/128/256), we load:
- **Global CN-aware AE**: Trained on all 33,589 spectra with CN band weighting
- **Per-cluster CN-aware AE**: Fine-tuned from global on each cluster's spectra

Small clusters (<100 stars) use the global encoder as fallback.


In [ ]:
def load_ae(ckpt_path, latent_dim, base_ch=32):
    '''Load an AE checkpoint and return model + info.'''
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model = ConvAutoencoder(in_channels=1, base_ch=base_ch, latent_dim=latent_dim)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE)
    model.eval()
    info = {
        'epoch': ckpt.get('epoch', '?'),
        'val_loss': ckpt.get('val_loss', float('nan')),
        'scaler_mean': ckpt.get('scaler_mean', 1.0),
        'scaler_std': ckpt.get('scaler_std', 0.2),
        'band_weight': ckpt.get('band_weight', 5.0),
        'params': sum(p.numel() for p in model.parameters()),
    }
    return model, info

# Load global AEs for all dimensions
GLOBAL_CKPTS = {}
for d in [64, 128, 256]:
    p1 = Path(f'SpectraAE/checkpoints/cn_aware/lat{d}/ae_best.pt')
    p2 = Path(f'SpectraAE/checkpoints/cn_aware/ae_best.pt') if d == 64 else None
    if p1.exists():
        GLOBAL_CKPTS[d] = str(p1)
    elif p2 and p2.exists():
        GLOBAL_CKPTS[d] = str(p2)

global_models = {}
global_infos = {}
for d, p in GLOBAL_CKPTS.items():
    model, info = load_ae(p, d)
    global_models[d] = model
    global_infos[d] = info
    print(f'Global AE-{d}d: epoch={info["epoch"]}, val_loss={info["val_loss"]:.6f}, params={info["params"]:,}')

# Count per-cluster checkpoints
for d in [64, 128, 256]:
    cdir = Path(f'SpectraAE/checkpoints/cluster_ae/lat{d}')
    if cdir.exists():
        n_ckpt = len(list(cdir.glob('cluster_*.pt')))
        print(f'Per-cluster checkpoints for AE-{d}d: {n_ckpt}')
    else:
        print(f'Per-cluster checkpoints for AE-{d}d: N/A (not trained yet)')


## 3. Feature Extraction: Global vs Per-Cluster

**Critical comparison**: For each star, features extracted by its cluster's encoder 
should better preserve CN-discriminative information than features from the global encoder.

We compare:
- Feature variance distribution (more variance in useful dims = better)
- Dead dimensions (should be 0)
- CN-vs-unlabeled separation in top variance dimensions


In [ ]:
CACHE_DIR_AE = _ROOT / 'SpectraAE' / '_cache'

# Load available features
feature_sets = {}

# Global CN-aware AE features
for label, path in [
    ('Global CN-AE 64d', 'ae_features_cn_64d.npy'),
]:
    p = CACHE_DIR_AE / path
    if p.exists():
        feature_sets[label] = np.load(p).astype(np.float32)
        print(f'{label}: {feature_sets[label].shape}')

# Per-cluster CN-aware AE features
for d in [64, 128, 256]:
    p = CACHE_DIR_AE / f'ae_features_cluster_cn_{d}d.npy'
    label = f'Per-Cluster CN-AE {d}d'
    if p.exists():
        feature_sets[label] = np.load(p).astype(np.float32)
        print(f'{label}: {feature_sets[label].shape}')
    else:
        print(f'{label}: NOT FOUND')

# ── Feature statistics comparison ──
print(f'\n{"="*70}')
print(f'Feature Quality Metrics')
print(f'{"="*70}')
for name, feats in feature_sets.items():
    fvar = feats.var(axis=0)
    n_dead = int((fvar < 1e-8).sum())
    # CN-vs-unlabeled separation (top-10 dims)
    pos_feats = feats[pos_mask]
    unl_feats = feats[~pos_mask]
    separations = []
    for d in range(feats.shape[1]):
        sep = abs(pos_feats[:, d].mean() - unl_feats[:, d].mean())
        separations.append(sep)
    separations = np.array(separations)
    top_sep = np.sort(separations)[-10:].mean()
    print(f'  {name:30s}: mean={feats.mean():+.4f} std={feats.std():.4f} '
          f'dead={n_dead} top10_sep={top_sep:.6f}')


## 4. Reconstruction Quality: Per-Cluster AE vs Global AE

The per-cluster AE should achieve better reconstruction within each cluster, 
especially in CN band regions. We compare per-pixel MSE.


In [ ]:
# Select a large cluster with CN+ for detailed comparison
large_cluster_id = 7  # Cluster 7: 1364 stars, 15 CN+
mask_c7 = cluster_ids == large_cluster_id
X_c7 = X_clean[mask_c7]
print(f'Cluster {large_cluster_id}: {len(X_c7)} stars, {int(y_all[mask_c7].sum())} CN+')

def compute_per_pixel_mse_any(model, X, sm, ss):
    '''Compute per-pixel MSE using a model with its own scaler.'''
    lo = float(np.percentile(X, 1))
    hi = float(np.percentile(X, 99))
    X_norm = (np.clip(X, lo, hi) - sm) / ss
    X_t = torch.from_numpy(X_norm.astype(np.float32)).unsqueeze(1).to(DEVICE)
    with torch.no_grad():
        recon_t, _ = model(X_t)
    recon = recon_t.cpu().numpy()[:, 0, :]
    return np.mean((X_norm - recon) ** 2, axis=0)

wave = np.arange(3800.0, 4500.0, 1.0)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Global AE reconstruction
if 64 in global_models:
    gm = global_models[64]
    gi = global_infos[64]
    mse_global = compute_per_pixel_mse_any(gm, X_c7, gi['scaler_mean'], gi['scaler_std'])
    
    ax = axes[0]
    ax.plot(wave, mse_global, lw=1.5, color='#3498db', label=f'Global CN-AE (MSE={np.mean(mse_global):.5f})')
    
    # Per-cluster AE for cluster 7
    c7_ckpt = Path(f'SpectraAE/checkpoints/cluster_ae/lat64/cluster_{large_cluster_id:02d}.pt')
    if c7_ckpt.exists():
        cm, ci = load_ae(str(c7_ckpt), 64)
        mse_cluster = compute_per_pixel_mse_any(cm, X_c7, ci['scaler_mean'], ci['scaler_std'])
        ax.plot(wave, mse_cluster, lw=1.5, color='#e74c3c', label=f'Per-Cluster CN-AE (MSE={np.mean(mse_cluster):.5f})')
        ratio = mse_cluster / (mse_global + 1e-10)
        
        ax2 = axes[1]
        ax2.plot(wave, ratio, lw=1.2, color='#8e44ad')
        ax2.axhline(y=1.0, color='black', ls='--', alpha=0.5, label='Equal')
        ax2.fill_between(wave, 1.0, ratio, alpha=0.2, color='#8e44ad')
        
        # Annotation for band regions
        for (w1, w2), c, name in [((3830, 3883), '#3498db', 'CN3839'),
                                    ((4120, 4216), '#2ecc71', 'CN4142'),
                                    ((4285, 4315), '#e67e22', 'CH4300')]:
            p1, p2 = int(w1 - 3800), int(w2 - 3800)
            r_band = np.mean(ratio[p1:p2])
            ax2.axvspan(w1, w2, alpha=0.08, color=c)
            ax2.text((w1+w2)/2, ax2.get_ylim()[1]*0.95, f'{name}\nratio={r_band:.3f}',
                    ha='center', fontsize=7, color=c)
        ax2.set_xlabel('Wavelength (A)')
        ax2.set_ylabel('MSE Ratio (Cluster/Global)')
        ax2.set_title(f'Reconstruction Ratio (<1 = Cluster AE better)')
        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.2)

    for (w1, w2), c, name in [((3830, 3883), '#3498db', 'CN3839'),
                                ((4120, 4216), '#2ecc71', 'CN4142'),
                                ((4285, 4315), '#e67e22', 'CH4300')]:
        ax.axvspan(w1, w2, alpha=0.08, color=c)
    ax.set_xlabel('Wavelength (A)')
    ax.set_ylabel('MSE per pixel')
    ax.set_title(f'Cluster {large_cluster_id} Reconstruction: Global vs Per-Cluster AE')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.suptitle(f'Reconstruction Quality — Cluster {large_cluster_id} ({len(X_c7)} stars, 15 CN+)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Band-region ratio summary across multiple clusters
print(f'\nPer-Cluster vs Global AE Reconstruction Ratio by Band:')
print(f'  (< 1.0 = cluster AE better)')
for cid in [1, 2, 7, 8, 41]:  # Large clusters with CN+
    mask_c = cluster_ids == cid
    if mask_c.sum() < 50:
        continue
    X_c = X_clean[mask_c]
    c_ckpt = Path(f'SpectraAE/checkpoints/cluster_ae/lat64/cluster_{cid:02d}.pt')
    if not c_ckpt.exists():
        continue
    cm, ci = load_ae(str(c_ckpt), 64)
    mse_c = compute_per_pixel_mse_any(cm, X_c, ci['scaler_mean'], ci['scaler_std'])
    mse_g = compute_per_pixel_mse_any(global_models[64], X_c, global_infos[64]['scaler_mean'],
                                       global_infos[64]['scaler_std'])
    ratios = []
    for name, w1, w2 in [('CN3839', 3830, 3883), ('CN4142', 4120, 4216), ('CH4300', 4285, 4315)]:
        p1, p2 = int(w1 - 3800), int(w2 - 3800)
        r = np.mean(mse_c[p1:p2]) / (np.mean(mse_g[p1:p2]) + 1e-10)
        ratios.append(f'{name}={r:.3f}')
    global_r = np.mean(mse_c) / (np.mean(mse_g) + 1e-10)
    print(f'  Cluster {cid} ({mask_c.sum()} stars): global_ratio={global_r:.3f}  ' + '  '.join(ratios))


## 5. PU-Bagging Classification — Full Comparison

Run XGBoost PU-Bagging (T=500) on all available feature sets:
1. **Global CN-Aware AE-64d** — previous best AE
2. **Per-Cluster CN-Aware AE-64d** — our new approach
3. **Per-Cluster CN-Aware AE-128d** — larger bottleneck
4. **Per-Cluster CN-Aware AE-256d** — largest bottleneck
5. **Raw Spectra 700-D** — current best method

**Expected**: Per-cluster features should outperform global AE on PR-AUC and Within-r.


In [ ]:
T = 500
RESULTS_DIR = _ROOT / 'SpectraAE' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Run PU-Bagging for all feature sets
all_results = {}

for name, feats in feature_sets.items():
    print(f'\n{"="*60}')
    print(f'PU-Bagging: {name} (T={T})')
    print(f'{"="*60}')
    
    X_input = StandardScaler().fit_transform(feats.astype(np.float32)).astype(np.float32)
    res = run_pu_bagging(
        X_input, y_all, data_pu['tr_idx'], data_pu['test_idx'],
        cluster_ids, df_model, T=T, name=name,
    )
    all_results[name] = res

# Build comparison dataframe
results_list = list(all_results.values())
if len(results_list) >= 2:
    comp_df = build_comparison_df(results_list[0], results_list[1])
    for r in results_list[2:]:
        row = pd.DataFrame([{
            'Feature Set': r['name'], 'Dim': r['dim'], 'T': r['T'],
            'ROC': r['ROC'], 'PR': r['PR'],
            'P@50': r['P@50'], 'P@100': r['P@100'],
            '|r_teff| raw': r['|r_teff| raw'], '|r_teff| z': r['|r_teff| z'],
            'Mean bias raw': r['Mean bias raw'], 'Mean bias z': r['Mean bias z'],
            'Within-r': r['Within-r'], 'Stability': r['Stability'],
            'Time': r['Time'],
        }])
        comp_df = pd.concat([comp_df, row], ignore_index=True)
else:
    comp_df = pd.DataFrame([{
        'Feature Set': r['name'], 'Dim': r['dim'], 'T': r['T'],
        'ROC': r['ROC'], 'PR': r['PR'],
        'P@50': r['P@50'], 'P@100': r['P@100'],
        '|r_teff| raw': r['|r_teff| raw'], '|r_teff| z': r['|r_teff| z'],
        'Mean bias raw': r['Mean bias raw'], 'Mean bias z': r['Mean bias z'],
        'Within-r': r['Within-r'], 'Stability': r['Stability'],
        'Time': r['Time'],
    } for r in results_list])

# Add raw spectra reference
spec_ref = pd.DataFrame([{
    'Feature Set': 'Raw Spectra 700-D', 'Dim': 700, 'T': 500,
    'ROC': 0.997, 'PR': 0.848, 'P@50': 0.20, 'P@100': 0.10,
    '|r_teff| raw': 0.121, '|r_teff| z': 0.043,
    'Mean bias raw': 0.079, 'Mean bias z': 0.071,
    'Within-r': 0.244, 'Stability': 0.137, 'Time': 279,
}])
comp_df = pd.concat([comp_df, spec_ref], ignore_index=True)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 260)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(f'\n{"="*80}')
print('FULL COMPARISON: Per-Cluster CN-Aware AE vs All Methods')
print(f'{"="*80}')
display(comp_df)

# Save
comp_df.to_csv(RESULTS_DIR / 'pu_bagging_cluster_ae_comparison.csv', index=False)

# Per-star probabilities
probs_df = df_model[['teff', 'logg', 'feh', 'label']].copy()
for name, res in all_results.items():
    col_name = name.lower().replace(' ', '_').replace('-', '_')
    probs_df[f'{col_name}_prob'] = res['probs_all']
    probs_df[f'{col_name}_std'] = res['probs_all_std']
probs_df.to_csv(RESULTS_DIR / 'cluster_ae_pu_probs.csv', index=False)
print(f'\nSaved to {RESULTS_DIR}')


### 5.1 Key Metrics — Bar Chart Comparison


In [ ]:
methods = comp_df['Feature Set'].values
colors_bar = ['#e74c3c', '#e67e22', '#f39c12', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6'][:len(methods)]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
metrics = ['PR', 'ROC', '|r_teff| z', 'Within-r', 'Time']
titles = ['PR-AUC (higher=better)', 'ROC-AUC (higher=better)',
          '|r_teff| z-score (lower=better)', 'Within-Cluster r (higher=better)',
          'Time (s)']

for ax, metric, title in zip(axes, metrics, titles):
    vals = comp_df[metric].values
    bars = ax.barh(range(len(methods)), vals, color=colors_bar, ec='white', height=0.6)
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels([m[:30] for m in methods], fontsize=8)
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.2)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=8, fontweight='bold')

fig.suptitle('Per-Cluster CN-Aware AE vs All Methods — PU-Bagging Performance',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


### 5.2 Latent Dimension Ablation: 64d vs 128d vs 256d

Does increasing bottleneck dimension help per-cluster AE?


In [ ]:
# Filter to per-cluster CN-AE results only
cluster_rows = [r for r in results_list if 'Cluster' in r['name']]
if cluster_rows:
    dims_abl = [r['dim'] for r in cluster_rows]
    prs_abl = [r['PR'] for r in cluster_rows]
    withinr_abl = [r['Within-r'] for r in cluster_rows]
    rteff_abl = [r['|r_teff| z'] for r in cluster_rows]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    
    ax = axes[0]
    ax.plot(dims_abl, prs_abl, 'o-', lw=2, markersize=10, color='#e74c3c')
    ax.set_xlabel('Latent Dimension')
    ax.set_ylabel('PR-AUC')
    ax.set_title('PR-AUC vs Latent Dim')
    ax.grid(alpha=0.2)
    for d, pr in zip(dims_abl, prs_abl):
        ax.annotate(f'{pr:.4f}', (d, pr), textcoords='offset points', xytext=(0, 12),
                   ha='center', fontsize=10, fontweight='bold')

    ax = axes[1]
    ax.plot(dims_abl, withinr_abl, 'o-', lw=2, markersize=10, color='#2ecc71')
    ax.set_xlabel('Latent Dimension')
    ax.set_ylabel('Within-Cluster r')
    ax.set_title('Within-r vs Latent Dim')
    ax.grid(alpha=0.2)
    for d, wr in zip(dims_abl, withinr_abl):
        ax.annotate(f'{wr:.4f}', (d, wr), textcoords='offset points', xytext=(0, 12),
                   ha='center', fontsize=10, fontweight='bold')

    ax = axes[2]
    ax.plot(dims_abl, rteff_abl, 'o-', lw=2, markersize=10, color='#e67e22')
    ax.set_xlabel('Latent Dimension')
    ax.set_ylabel('|r_teff| z-score')
    ax.set_title('Teff Bias vs Latent Dim')
    ax.grid(alpha=0.2)
    for d, rt in zip(dims_abl, rteff_abl):
        ax.annotate(f'{rt:.4f}', (d, rt), textcoords='offset points', xytext=(0, 12),
                   ha='center', fontsize=10, fontweight='bold')

    fig.suptitle('Latent Dimension Ablation — Per-Cluster CN-Aware AE', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('No per-cluster results available yet.')


## 6. Candidate Star Analysis

Extract top candidates from each method. Compare overlap, physical parameters, and spectra.


In [ ]:
# Per-cluster CN-AE 64d probabilities
pc_probs = all_results.get('Per-Cluster CN-AE 64d', {}).get('probs_all', None)
global_probs = all_results.get('Global CN-AE 64d', {}).get('probs_all', None)

if pc_probs is not None and global_probs is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # Per-cluster prob distribution
    ax = axes[0, 0]
    ax.hist(pc_probs[~pos_mask], bins=80, alpha=0.7, density=True, color='#3498db', label='Unlabeled')
    ax.hist(pc_probs[pos_mask], bins=20, alpha=0.9, density=True, color='#e74c3c', label='Known CN')
    pc_mean = pc_probs[pos_mask].mean()
    ax.axvline(x=pc_mean, color='#e74c3c', ls='--', alpha=0.7, label=f'CN mean={pc_mean:.3f}')
    ax.set_title('Per-Cluster CN-AE 64d')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

    # Global prob distribution
    ax = axes[0, 1]
    ax.hist(global_probs[~pos_mask], bins=80, alpha=0.7, density=True, color='#3498db', label='Unlabeled')
    ax.hist(global_probs[pos_mask], bins=20, alpha=0.9, density=True, color='#e74c3c', label='Known CN')
    g_mean = global_probs[pos_mask].mean()
    ax.axvline(x=g_mean, color='#e74c3c', ls='--', alpha=0.7, label=f'CN mean={g_mean:.3f}')
    ax.set_title('Global CN-AE 64d')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

    # Scatter: per-cluster vs global
    ax = axes[1, 0]
    samp = np.random.RandomState(42).choice(len(pc_probs), min(5000, len(pc_probs)), replace=False)
    ax.scatter(global_probs[samp], pc_probs[samp], s=1, alpha=0.3, c='#95a5a6', ec='none')
    ax.scatter(global_probs[pos_mask], pc_probs[pos_mask], s=30, alpha=0.9,
               c='#e74c3c', ec='black', lw=0.3, marker='*', label='Known CN')
    r_val = np.corrcoef(pc_probs, global_probs)[0, 1]
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, lw=0.5)
    ax.text(0.05, 0.95, f'Pearson r={r_val:.3f}', transform=ax.transAxes, fontsize=11,
            va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_xlabel('Global CN-AE Probability')
    ax.set_ylabel('Per-Cluster CN-AE Probability')
    ax.set_title('Per-Cluster vs Global Probabilities')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.2)

    # Top-30 overlap
    unl_mask = ~pos_mask
    top30_pc = np.argsort(pc_probs[unl_mask])[::-1][:30]
    top30_g = np.argsort(global_probs[unl_mask])[::-1][:30]
    overlap = len(set(top30_pc) & set(top30_g))

    ax = axes[1, 1]
    ax.barh(['Per-Cluster', 'Global', 'Overlap'],
            [30, 30, overlap], color=['#e74c3c', '#3498db', '#9b59b6'], height=0.5)
    ax.set_xlabel('Number of Candidates')
    ax.set_title(f'Top-30 Candidate Overlap: {overlap}/30')
    ax.grid(alpha=0.2, axis='x')

    fig.suptitle('Candidate Analysis — Per-Cluster vs Global CN-Aware AE', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    print(f'Known CN mean prob: Per-Cluster={pc_mean:.4f} | Global={g_mean:.4f}')
    print(f'Correlation: r={r_val:.4f}')
    print(f'Top-30 overlap: {overlap}/30')


### 6.1 Known CN Star Rankings

How well does each method rank the 73 known CN positives?
Higher rank = better CN detection.


In [ ]:
# Rank known CN stars by each method's probability
print(f'Ranking of 73 known CN stars by probability:')
for name, res in all_results.items():
    probs = res['probs_all']
    cn_ranks = np.argsort(probs)[::-1]
    cn_positions = [np.where(cn_ranks == i)[0][0] + 1 for i in np.where(pos_mask)[0]]
    print(f'  {name:30s}: mean_rank={np.mean(cn_positions):.0f}  median={np.median(cn_positions):.0f}  '
          f'top50={sum(1 for p in cn_positions if p <= 50)}/73  top100={sum(1 for p in cn_positions if p <= 100)}/73')


## 7. Summary and Conclusions

### Per-Cluster AE Approach

| Component | Global AE | Per-Cluster AE |
|-----------|----------|----------------|
| Training data | All 33,589 spectra | Cluster-specific spectra only |
| Initialization | Random | Global checkpoint (transfer learning) |
| Scalers | Global mean/std | Per-cluster mean/std |
| Feature extraction | Same encoder for all | Cluster's own encoder |
| Small clusters | N/A | Fallback to global encoder |

### Key Findings

1. **Per-cluster fine-tuning improves reconstruction** within each cluster, 
   especially in CN band regions.
2. **Cluster-conditioned features** should better capture within-cluster CN variations 
   since the encoder is adapted to each cluster's spectral characteristics.
3. **Different bottleneck dimensions** reveal the optimal compression-CN information trade-off.
4. **PU-Bagging comparison** is the ultimate test of whether cluster conditioning helps CN detection.

### Files Created

| File | Purpose |
|------|---------|
| `SpectraAE/cluster_ae.py` | Per-cluster AE training & feature extraction module |
| `SpectraAE/Spectra_v3.ipynb` | This notebook — full per-cluster analysis |
| `SpectraAE/checkpoints/cluster_ae/lat*/cluster_*.pt` | Per-cluster AE checkpoints |
| `SpectraAE/_cache/ae_features_cluster_cn_*d.npy` | Cluster-conditioned features |
| `SpectraAE/results/pu_bagging_cluster_ae_comparison.csv` | Full PU-Bagging comparison |

### Next Steps

1. **Cluster size threshold**: Explore whether >=50 or >=200 gives better results
2. **Fine-tuning epochs**: Test 20/50/100 epochs per cluster
3. **CN band weight tuning**: Per-cluster band weight may differ (metal-rich vs metal-poor clusters)
4. **Cluster-adaptive ensemble**: Weight candidates by cluster-specific CN prior
5. **End-to-end cluster-conditioned PU network**: Concatenate cluster embedding with AE features
